# Chuẩn hóa dữ liệu VG Sales

## Mục lục
1. Chọn dataset
2. Dataset có gì
3. Đánh giá vấn đề
4. Đưa ra giải pháp
5. Kiểm chứng

In [ ]:
# Cài đặt các thư viện cần thiết (chỉ cần chạy 1 lần)
%pip install pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Đọc dữ liệu
df = pd.read_csv("vgsales.csv")
print(f"Kích thước: {df.shape[0]} dòng x {df.shape[1]} cột")
df.head()

## 1. Chọn Dataset

- **Tên:** Video Game Sales
- **Nguồn:** Kaggle
- **Mô tả:** Dữ liệu doanh số các game trên toàn thế giới
- **Số dòng:** ~16,598
- **Số cột:** 11

## 2. Dataset có gì?

In [ ]:
print("=== Tên cột và kiểu dữ liệu ===")
print(df.dtypes)
print("\n=== Thông tin tổng quan ===")
df.info()

In [ ]:
print("=== 5 dòng đầu ===")
display(df.head())
print("\n=== 5 dòng cuối ===")
display(df.tail())

In [ ]:
print("=== Giá trị thiếu ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Số thiếu': missing, 'Tỷ lệ %': missing_pct.round(2)})
display(missing_df)

In [ ]:
# Biểu đồ cột: số lượng giá trị thiếu theo từng cột
missing_nonzero = missing_df[missing_df['Số thiếu'] > 0].sort_values('Số thiếu', ascending=False)

fig, ax = plt.subplots()
bars = ax.bar(missing_nonzero.index, missing_nonzero['Số thiếu'], color='#e74c3c', width=0.4)
ax.set_title('Số lượng giá trị thiếu theo cột')
ax.set_xlabel('Cột')
ax.set_ylabel('Số dòng thiếu')
ax.bar_label(bars, padding=3)
ax.set_xlim(-0.5, len(missing_nonzero) - 0.5)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Dòng trùng lặp ===")
print(f"Dòng trùng hoàn toàn: {df.duplicated().sum()}")
print(f"Tên game trùng: {df['Name'].duplicated().sum()}")

print("\n=== Số lượng unique values ===")
for col in df.columns:
    print(f"{col}: {df[col].nunique()}")

## 3. Đánh giá vấn đề

In [ ]:
print("=== Vấn đề 1: Year có giá trị NaN hoặc ngoài phạm vi ===")
print(f"Year NaN: {df['Year'].isna().sum()}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
invalid_year = df[(df['Year'] < 1980) | (df['Year'] > 2025)]
print(f"Year ngoài [1980, 2025]: {len(invalid_year)}")
if len(invalid_year) > 0:
    display(invalid_year.head())

In [ ]:
# Biểu đồ cột: phân bố số game theo Year (làm nổi bật vùng ngoài [1980, 2025])
year_counts = df['Year'].value_counts().sort_index()
colors = ['#e74c3c' if (y < 1980 or y > 2025) else '#3498db' for y in year_counts.index]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(year_counts.index.astype(str), year_counts.values, color=colors, width=0.7)
ax.set_title('Số lượng game theo Year (đỏ = ngoài phạm vi hợp lệ [1980, 2025])')
ax.set_xlabel('Year')
ax.set_ylabel('Số game')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
print("=== Vấn đề 2: Publisher có giá trị thiếu ===")
print(f"Publisher NaN: {df['Publisher'].isna().sum()}")

print("\n=== Vấn đề 3: Cột Sales có giá trị âm ===")
for col in ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']:
    neg_count = (df[col] < 0).sum()
    print(f"{col} < 0: {neg_count}")

print("\n=== Vấn đề 4: Rank bị trùng hoặc thiếu ===")
print(f"Rank trùng: {df['Rank'].duplicated().sum()}")
print(f"Rank thiếu: {df['Rank'].isna().sum()}")

In [ ]:
# Biểu đồ cột: tổng hợp số dòng bị ảnh hưởng bởi từng vấn đề
issues_summary = {
    'Year NaN': df['Year'].isna().sum(),
    'Year ngoài [1980,2025]': len(invalid_year),
    'Publisher NaN': df['Publisher'].isna().sum(),
    'Sales âm (tổng)': sum((df[c] < 0).sum() for c in ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']),
    'Rank trùng': df['Rank'].duplicated().sum(),
    'Rank thiếu': df['Rank'].isna().sum(),
}

fig, ax = plt.subplots()
bars = ax.bar(issues_summary.keys(), issues_summary.values(), color='#f39c12', width=0.5)
ax.set_title('Tổng hợp số dòng bị ảnh hưởng theo từng vấn đề')
ax.set_ylabel('Số dòng')
ax.bar_label(bars, padding=3)
ax.set_xlim(-0.5, len(issues_summary) - 0.5)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Vấn đề 5: Name có giá trị trùng (cùng game, khác platform) ===")
dup_names = df.groupby('Name').size().sort_values(ascending=False)
print(f"Tổng số game trùng tên: {(dup_names > 1).sum()}")
print("\nTop 5 game xuất hiện nhiều nhất:")
print(dup_names.head(10))

print("\n=== Vấn đề 6: Platform và Genre ===")
print(f"Số Platform unique: {df['Platform'].nunique()}")
print(f"Số Genre unique: {df['Genre'].nunique()}")
print(f"\nPlatforms: {sorted(df['Platform'].unique())}")
print(f"\nGenres: {sorted(df['Genre'].unique())}")

In [ ]:
# Biểu đồ cột: Top 10 game trùng tên nhiều nhất
top_dup = dup_names.head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_dup.index[::-1], top_dup.values[::-1], color='#9b59b6', height=0.5)
ax.set_title('Top 10 game trùng tên nhiều nhất (xuất hiện trên nhiều Platform)')
ax.set_xlabel('Số lần xuất hiện')
ax.bar_label(bars, padding=3)
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ cột: số lượng game theo Platform và theo Genre
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

platform_counts = df['Platform'].value_counts()
axes[0].bar(platform_counts.index, platform_counts.values, color='#1abc9c', width=0.6)
axes[0].set_title('Số lượng game theo Platform')
axes[0].set_xlabel('Platform')
axes[0].set_ylabel('Số game')
axes[0].tick_params(axis='x', rotation=90)

genre_counts = df['Genre'].value_counts()
axes[1].bar(genre_counts.index, genre_counts.values, color='#2ecc71', width=0.6)
axes[1].set_title('Số lượng game theo Genre')
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Số game')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Giải pháp

In [ ]:
# Tạo bản sao để xử lý
df_clean = df.copy()
print(f"Trước clean: {df_clean.shape}")

In [ ]:
# 4.1: Xóa dòng trùng lặp hoàn toàn
df_clean = df_clean.drop_duplicates()
print(f"Sau xóa trùng: {df_clean.shape}")

In [ ]:
# 4.2: Xử lý Year - điền NaN bằng giá trị phổ biến nhất
year_mode = df_clean['Year'].mode()[0]
print(f"Year mode: {year_mode}")
df_clean['Year'] = df_clean['Year'].fillna(year_mode).astype(int)

# Xóa dòng Year ngoài phạm vi hợp lý [1980, 2020]
invalid_year_mask = (df_clean['Year'] < 1980) | (df_clean['Year'] > 2020)
print(f"Year ngoài [1980, 2020]: {invalid_year_mask.sum()}")
df_clean = df_clean[~invalid_year_mask]

In [ ]:
# 4.3: Xử lý Publisher - điền NaN bằng 'Unknown'
df_clean['Publisher'] = df_clean['Publisher'].fillna('Unknown')
print(f"Publisher NaN còn lại: {df_clean['Publisher'].isna().sum()}")

In [ ]:
# 4.4: Kiểm tra và xóa dòng có Sales âm
for col in ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']:
    neg_mask = df_clean[col] < 0
    if neg_mask.sum() > 0:
        print(f"Xóa {neg_mask.sum()} dòng có {col} < 0")
        df_clean = df_clean[~neg_mask]

In [ ]:
# 4.5: Sửa lại Rank sau khi xóa dòng
df_clean = df_clean.sort_values('Global_Sales', ascending=False).reset_index(drop=True)
df_clean['Rank'] = range(1, len(df_clean) + 1)
print(f"Rank mới: 1 đến {df_clean['Rank'].max()}")

In [ ]:
# 4.6: Chuẩn hóa chuỗi (xóa khoảng trắng thừa)
for col in df_clean.select_dtypes(include=['object', 'str']).columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# 4.7: Làm tròn cột Sales
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
df_clean[sales_cols] = df_clean[sales_cols].round(2)

print(f"\nKích thước sau clean: {df_clean.shape}")

## 5. Kiểm chứng

In [ ]:
print("=== KIỂM CHỨNG SAU CLEAN ===")

# 5.1: Không còn dòng trùng
print(f"\n1. Dòng trùng hoàn toàn: {df_clean.duplicated().sum()} (yêu cầu: 0)")

# 5.2: Year không còn NaN, trong phạm vi [1980, 2020]
print(f"2. Year NaN: {df_clean['Year'].isna().sum()}")
print(f"   Year range: [{df_clean['Year'].min()}, {df_clean['Year'].max()}]")

# 5.3: Publisher không còn NaN
print(f"3. Publisher NaN: {df_clean['Publisher'].isna().sum()}")

# 5.4: Sales không còn âm
for col in sales_cols:
    neg_count = (df_clean[col] < 0).sum()
    print(f"4. {col} < 0: {neg_count}")

In [ ]:
# 5.5: Rank liên tục từ 1
print(f"5. Rank: {df_clean['Rank'].min()} đến {df_clean['Rank'].max()}, liên tục: {df_clean['Rank'].nunique() == len(df_clean)}")

# 5.6: Kiểm tra Global_Sales = tổng các cột khác
df_clean['Calc_Global'] = df_clean[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].sum(axis=1).round(2)
match = (df_clean['Calc_Global'] == df_clean['Global_Sales']).sum()
print(f"6. Global_Sales = NA+EU+JP+Other: {match}/{len(df_clean)} khớp")
df_clean = df_clean.drop('Calc_Global', axis=1)

In [ ]:
print("=== Giá trị thiếu còn lại ===")
missing_final = df_clean.isnull().sum()
print(missing_final[missing_final > 0] if missing_final.sum() > 0 else "Không có giá trị thiếu!")

In [ ]:
# 5.7: So sánh trước/sau
print("=== SO SÁNH TRƯỚC/SAU ===")
print(f"{'Chỉ số':<25}{'Trước':>15}{'Sau':>15}")
print("-" * 55)
print(f"{'Số dòng':<25}{len(df):>15}{len(df_clean):>15}")
print(f"{'Dòng trùng':<25}{df.duplicated().sum():>15}{df_clean.duplicated().sum():>15}")
print(f"{'Year NaN':<25}{df['Year'].isna().sum():>15}{df_clean['Year'].isna().sum():>15}")
print(f"{'Publisher NaN':<25}{df['Publisher'].isna().sum():>15}{df_clean['Publisher'].isna().sum():>15}")

In [ ]:
# Biểu đồ cột: so sánh Trước/Sau clean cho từng chỉ số
compare_data = {
    'Dòng trùng': (df.duplicated().sum(), df_clean.duplicated().sum()),
    'Year NaN': (df['Year'].isna().sum(), df_clean['Year'].isna().sum()),
    'Publisher NaN': (df['Publisher'].isna().sum(), df_clean['Publisher'].isna().sum()),
}

labels = list(compare_data.keys())
before_vals = [v[0] for v in compare_data.values()]
after_vals = [v[1] for v in compare_data.values()]

x = np.arange(len(labels))
width = 0.3

fig, ax = plt.subplots()
bars1 = ax.bar(x - width/2, before_vals, width, label='Trước', color='#e74c3c')
bars2 = ax.bar(x + width/2, after_vals, width, label='Sau', color='#27ae60')
ax.set_title('So sánh các chỉ số Trước/Sau khi clean dữ liệu')
ax.set_ylabel('Số lượng')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.bar_label(bars1, padding=3)
ax.bar_label(bars2, padding=3)
ax.set_xlim(-0.5, len(labels) - 0.5)
plt.tight_layout()
plt.show()

# Biểu đồ cột: số dòng trước/sau clean
fig, ax = plt.subplots()
bars = ax.bar(['Trước clean', 'Sau clean'], [len(df), len(df_clean)], color=['#e74c3c', '#27ae60'], width=0.4)
ax.set_title('Số dòng dữ liệu: Trước vs Sau khi clean')
ax.set_ylabel('Số dòng')
ax.bar_label(bars, padding=3)
ax.set_xlim(-0.5, 1.5)
plt.tight_layout()
plt.show()

In [ ]:
# Lưu file đã clean
df_clean.to_csv("vgsales_cleaned.csv", index=False)
print("✅ Đã lưu: vgsales_cleaned.csv")
display(df_clean.head(10))